# Combined Recommendation System

combining multiple recommenders with weighted scores. easy to add new ones later

In [1]:
%load_ext autoreload
%autoreload 2

## setup and data loading

first connect to db and load all the data we need

In [2]:
import os
import asyncio
import asyncpg
from dotenv import load_dotenv
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')
if (DATABASE_URL is None):
    raise ValueError("DATABASE_URL is not set in environment variables")


def remove_schema_param(url):
    """Remove the schema parameter from DATABASE_URL for asyncpg"""
    parsed = urlparse(url)
    query_params = parse_qs(parsed.query)

    # Remove 'schema' parameter if it exists
    query_params.pop('schema', None)

    # Rebuild the URL without schema parameter
    new_query = urlencode(query_params, doseq=True)
    new_parsed = parsed._replace(query=new_query)
    return urlunparse(new_parsed)

asyncpg_url = remove_schema_param(DATABASE_URL)
conn = await asyncpg.connect(dsn=asyncpg_url)

setup

load the FAISS index and embeddings for finding similar events

In [3]:
import pandas as pd
import faiss
import numpy as np
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import pickle
import math

from utils import load_events_df, load_user_event_associations


index_path = "event_embeddings_shard_"
meta_path = "models/event_index_meta.pkl"
model_name_label = "all-MiniLM-L6-v2"

user_event_associations_df = await load_user_event_associations(conn)

print(f"Loaded {len(user_event_associations_df):,} user-event rows")

# get combined events with caching
events_df = await load_events_df(conn)
print(f"Loaded {len(events_df)} rows into events_df")

shard_paths = sorted(
    os.path.join("models", p)
    for p in os.listdir("models")
    if p.startswith(index_path)
)

have_shards = len(shard_paths) > 0

if have_shards and os.path.exists(meta_path):
    print("Loading FAISS index and metadata from disk...")

    # load FAISS index
    indices = []
    shard_paths = sorted(p for p in os.listdir("models") if p.startswith("event_embeddings_shard_"))

    for p in shard_paths:
        indices.append(faiss.read_index(os.path.join("models", p)))

    d = indices[0].d
    index = faiss.IndexFlatIP(d)

    def get_vectors(idx: faiss.Index) -> np.ndarray:
        """Return all vectors stored in a flat FAISS index as (n, d) float32 array."""
        n = idx.ntotal
        if hasattr(idx, "reconstruct_n"):
            return idx.reconstruct_n(0, n)
        # fallback: reconstruct one by one if reconstruct_n is missing
        return np.vstack(idx.reconstruct(i) for i in range(n))

    for shard_id, idx in enumerate(indices):
        xb = get_vectors(idx).astype("float32")
        index.add(xb)
        print(f"Added shard {shard_id} with {idx.ntotal} vectors")

    print("Merged index ntotal:", index.ntotal)

    # load mappings / metadata
    with open(meta_path, "rb") as f:
        meta = pickle.load(f)

    index_to_id = meta["index_to_id"]
    id_to_index = meta["id_to_index"]
    model_name = meta.get("model_name", model_name_label)

    # reload embedding model
    embedding_model = SentenceTransformer(model_name)

    if hasattr(index, "reconstruct_n"):
        normalized_embeddings = index.reconstruct_n(0, index.ntotal)  # shape (n, d)
    else:
        # slower fallback
        normalized_embeddings = np.vstack(index.reconstruct(i) for i in range(n))

    print(normalized_embeddings.shape)
else:
    print("No saved index found. Building FAISS index from scratch...")
    topic_model = BERTopic(embedding_model=model_name_label)

    # Create combined text for embeddings
    events_df["combined_text"] = (
        events_df["title"].fillna("") + " " + events_df["description"].fillna("")
    )

    embedding_model = SentenceTransformer(model_name_label)
    embeddings = embedding_model.encode(
        events_df["combined_text"].tolist(), show_progress_bar=True
    )

    embedding_dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(embedding_dim)

    normalized_embeddings = embeddings / np.linalg.norm(
        embeddings, axis=1, keepdims=True
    )
    index.add(normalized_embeddings.astype("float32"))

    index_to_id = {i: id for i, id in enumerate(events_df.index)}
    id_to_index = {id: i for i, id in enumerate(events_df.index)}

    # Save FAISS index
    emb = normalized_embeddings.astype("float32")
    n, d = emb.shape
    shard_size = 15000  # adjust to keep each file <100MB

    n_shards = math.ceil(n / shard_size)

    for shard_id in range(n_shards):
        start = shard_id * shard_size
        end = min(start + shard_size, n)
        part = emb[start:end]

        idx = faiss.IndexFlatIP(d)
        idx.add(part)

        faiss.write_index(idx, f"models/event_embeddings_shard_{shard_id:02d}.index")

    # Save mappings
    meta = {
        "index_to_id": index_to_id,
        "id_to_index": id_to_index,
        "model_name": model_name_label,
    }

    with open(meta_path, "wb") as f:
        pickle.dump(meta, f)

c:\Users\euseb\anaconda3\envs\t2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading from 14 parquet files in 'data/user_event_associations_cache/'...
Loaded 483,767 user-event rows
Loading from 3 parquet files in 'data/events_df_cache/'...
Loaded 65056 rows into events_df
Loading FAISS index and metadata from disk...
Added shard 0 with 15000 vectors
Added shard 1 with 15000 vectors
Added shard 2 with 15000 vectors
Added shard 3 with 15000 vectors
Added shard 4 with 15000 vectors
Added shard 5 with 15000 vectors
Added shard 6 with 8450 vectors
Merged index ntotal: 98450
(98450, 384)


In [4]:
from collaborative_filtering.cf_recommender import load_cf_model

load_cf_model()

Loading CF model...


c:\Users\euseb\anaconda3\envs\t2\Lib\site-packages\implicit\cpu\als.py:95: RuntimeWarning: OpenBLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


Loading from 82 parquet files in 'collaborative_filtering/df_trade_tags_cache/'...
  8579 users, 3666 tags
Loading from 7 parquet files in 'collaborative_filtering/events_tag_cache/'...
  65056 events mapped
done


In [19]:
from recommendation_system.main import build_and_cache_association_rules

build_and_cache_association_rules(force_rebuild=False)

Building association rules from scratch...
Fetching all user transactions...
STEP 1: FETCHING TRANSACTION DATA

TRANSACTION STATISTICS:
	Total transactions: 1972
	Average items per transaction: 46.16
	Total unique items: 37323
Generating frequent itemsets...

Minimum support threshold: 0.01 (1.0%)
Generating association rules...
STEP 6: GENERATING ASSOCIATION RULES

Minimum confidence: 0.3 (30.0%)
Minimum lift: 1.1
Caching rules to CSV...
Rules exported to: data/association_cache/association_rules_20251202_161331.csv
APRIORI ANALYSIS SUMMARY

TRANSACTIONS:
	Total: 1972
	Avg items: 46.16

FREQUENT ITEM SETS:
	Total: 496
	1-item_sets: 392
	2-item_sets: 102
	3-item_sets: 2

ASSOCIATION RULES:
	Total: 115
	Avg confidence: 0.493
	Avg lift: 8.024
	Max lift: 24.415

TOP 50 RULES (by lift):
	1. over-1pt2b-committed-to-the-megaeth-public-sale-863_Yes → over-1b-committed-to-the-megaeth-public-sale_Yes
		Confidence: 61.9%, Lift: 24.42
	2. over-1b-committed-to-the-megaeth-public-sale_Yes → over-1p

,antecedent,consequent,support,confidence,lift,conviction,antecedent_list,consequent_list
0,over-1pt2b-committed-to-the-megaeth-public-sal...,over-1b-committed-to-the-megaeth-public-sale_Yes,0.012981,0.619048,24.799048,2.559473,[over-1pt2b-committed-to-the-megaeth-public-sa...,[over-1b-committed-to-the-megaeth-public-sale_...
1,over-1b-committed-to-the-megaeth-public-sale_Yes,over-1pt2b-committed-to-the-megaeth-public-sal...,0.012981,0.520000,24.799048,2.039649,[over-1b-committed-to-the-megaeth-public-sale_...,[over-1pt2b-committed-to-the-megaeth-public-sa...
2,over-1pt8b-committed-to-the-megaeth-public-sal...,over-1pt4b-committed-to-the-megaeth-public-sal...,0.012981,0.619048,23.845238,2.556852,[over-1pt8b-committed-to-the-megaeth-public-sa...,[over-1pt4b-committed-to-the-megaeth-public-sa...
3,over-1pt4b-committed-to-the-megaeth-public-sal...,over-1pt8b-committed-to-the-megaeth-public-sal...,0.012981,0.500000,23.845238,1.958063,[over-1pt4b-committed-to-the-megaeth-public-sa...,[over-1pt8b-committed-to-the-megaeth-public-sa...
4,will-the-party-for-freedom-win-the-second-most...,will-democrats-66-win-the-most-seats-in-the-20...,0.010484,0.677419,23.804754,3.011782,[will-the-party-for-freedom-win-the-second-mos...,[will-democrats-66-win-the-most-seats-in-the-2...
...,...,...,...,...,...,...,...,...
100,will-the-los-angeles-dodgers-win-the-2025-worl...,will-zohran-mamdani-win-the-2025-nyc-mayoral-e...,0.011483,0.359375,2.277937,1.314711,[will-the-los-angeles-dodgers-win-the-2025-wor...,[will-zohran-mamdani-win-the-2025-nyc-mayoral-...
101,will-the-government-shutdown-end-november-16-o...,will-zohran-mamdani-win-the-2025-nyc-mayoral-e...,0.013480,0.355263,2.251874,1.306326,[will-the-government-shutdown-end-november-16-...,[will-zohran-mamdani-win-the-2025-nyc-mayoral-...
102,us-x-venezuela-military-engagement-by-october-...,will-zohran-mamdani-win-the-2025-nyc-mayoral-e...,0.016475,0.351064,2.225256,1.297873,[us-x-venezuela-military-engagement-by-october...,[will-zohran-mamdani-win-the-2025-nyc-mayoral-...
103,xi-jinping-out-in-2025_No,will-zohran-mamdani-win-the-2025-nyc-mayoral-e...,0.017474,0.333333,2.112869,1.263355,[xi-jinping-out-in-2025_No],[will-zohran-mamdani-win-the-2025-nyc-mayoral-...


Here is our combined recommender:

In [ ]:
from topic_based_recommender.recommend_events_for_user_by_topic_based import recommend_events_for_user_by_topic_based
from recommendation_system.main import get_recommendations_for_user_fast
from collaborative_filtering.cf_recommender import recommend_events


def combined_recommendation_for_user(user_address, filter_traded=True, filter_expired=True):
    top_n = 10
    similar_per_event = 5
    
    topic_recs = recommend_events_for_user_by_topic_based(
        id_to_index, normalized_embeddings, index, index_to_id,
        user_event_associations_df, events_df,
        user_address, top_n, similar_per_event
    )
    
    assoc_recs = get_recommendations_for_user_fast(user_address)
    
    cf_recs = recommend_events(user_address, top_n)
    
    return topic_recs, assoc_recs, cf_recs

In [6]:
from collaborative_filtering.cf_recommender import get_cf_users

topic_users = set(user_event_associations_df['address'].unique())
cf_users = get_cf_users()

assoc_query = """
    SELECT DISTINCT up.address
    FROM "UserTrade" ut
    JOIN "UserProfile" up ON ut."proxyWallet" = up."proxyWallet"
    WHERE up.address IS NOT NULL
"""
assoc_rows = await conn.fetch(assoc_query)
assoc_users = set(row['address'] for row in assoc_rows)

common = topic_users & cf_users & assoc_users

print(f"Topic-based users:\t{len(topic_users):,}")
print(f"Association users:\t{len(assoc_users):,}")
print(f"CF users:\t{len(cf_users):,}")
print(f"Combined:\t{len(common):,}")

Topic-based users:	24,640
Association users:	8,584
CF users:	8,579
Combined:	8,579


In [22]:
example_user = list(common)[5]
reccomandation_systems = combined_recommendation_for_user(example_user)

Found proxyWallet for '0x47d453d758968b6858e64fb806d491d2f250e851': 0xfc92e2036b3b4a3621dd074bc240030fed60408c
FETCHING TRANSACTION DATA FOR SINGLE USER
Raw rows fetched for user: 0
User '0x47d453d758968b6858e64fb806d491d2f250e851' transaction statistics:
	Total transactions (after min_items=1): 0


In [ ]:
if reccomandation_systems:
    topic_recs = reccomandation_systems[0]
    association_recs = reccomandation_systems[1]
    cf_recs = reccomandation_systems[2]
    
    print(f"Recommendations for: {example_user}")
    
    print("Topic-Based:")
    if topic_recs:
        for rank, rec in enumerate(topic_recs, 1):
            print(f"{rank}. [{rec['id']}] {rec['title']}")

    print("Association Rules:")
    if association_recs:
        for rank, rec in enumerate(association_recs, 1):
            print(f"\n   {rank}. {rec['recommended_market']}")
            print(f"\tBased on: {rec['based_on']}")
            print(f"\tConfidence: {rec['confidence']:.1%} (probability)")
            print(f"\tLift: {rec['lift']:.2f}x (correlation strength)")
            print(f"\tSupport: {rec['support']:.1%} (frequency)")


    print("Collaborative Filtering:")
    if cf_recs:
        for rank, rec in enumerate(cf_recs, 1):
            print(f"  {rank}. [{rec.get('score',0):.2f}] {rec.get('title','?')[:50]}")

Recommendations for: 0x47d453d758968b6858e64fb806d491d2f250e851
Topic-Based:
1. [11765] Will Kamala announce VP pick today?
2. [11774] Iran military response by Friday?
3. [11240] Trump to speak at Bitcoin 2024 conference?
4. [11837] Will Kamala announce VP pick on Monday?
5. [13998] Solana above $170 on November 8?
6. [11952] Solana above $150 on August 16?
7. [12081] How many times will Trump tweet by next Friday?
8. [28623] 2025 July hottest on record?
9. [11303] Ethereum above $3,500 on July 5?
10. [11673] Biden seen in public today? 
Association Rules:
Collaborative Filtering:
  1. [1.03] Biden's dropout speech over 15 minutes?
  2. [1.03] Biden stumbles again by Friday?
  3. [1.03] Biden stumbles by Friday?
  4. [1.03] Will Biden still have a cold next week?
  5. [1.03] Biden cognitive test in July?
  6. [1.03] Biden resigns from presidency by August 31?
  7. [1.03] Biden dropout letter published without consent?
  8. [1.03] Will Biden's 538 approval reach 40% in July?
  9. [1.03